In [1]:
# !pip install --upgrade openai pydantic

In [2]:
import openai
import os
from pydantic import BaseModel
from typing import List

from gm_instruction_generator import GM_InstructionGenerator
from gm_json_generator import GM_JSONGenerator
from vm_logic_validator import VM_LogicValidator
from vm_syntax_validator import VM_SyntaxValidator
from gm_sample_picker import SamplePicker
from ds_system_prompts import SystemPromptRepository

os.environ['OPENAI_API_KEY'] = '***REMOVED***'

class DataSpinner:
    def __init__(self, seed_folder: str):
        """
        Initialize the DataSpinner with a seed folder. Modules and prompts are configured here.
        """
        self.picker = SamplePicker(seed_folder=seed_folder)
        self.instruction_gen = GM_InstructionGenerator("gpt-4.1")
        self.json_gen = GM_JSONGenerator("gpt-4.1")
        self.logic_val = VM_LogicValidator("gpt-4.1")
        self.syntax_val = VM_SyntaxValidator("gpt-4.1")
        self.supervisor = VM_Supervisor("gpt-4.1")

        # load system prompts
        self.prompt_instr = SystemPromptRepository.get_instruction_generation_prompt()
        self.prompt_json = SystemPromptRepository.get_json_generation_prompt()
        self.prompt_logic = SystemPromptRepository.get_logic_validation_prompt()
        self.prompt_syntax = SystemPromptRepository.get_syntax_validation_prompt()
        self.prompt_sup = SystemPromptRepository.get_supervisor_prompt()

    def run_once(self, seed_type: str, batch_size: int, num_examples: int) -> int:
        """
        Execute one iteration of the pipeline for a given seed_type:
          1) sample examples
          2) generate instructions
          3) convert to JSON
          4) logic & syntax validate
          5) supervise and keep only valid
          6) append valid to seed
        Returns the number of new pairs appended.
        """
        # 1) sample seed examples
        data_sample = self.picker.get_full_examples(seed_type, num_examples)
        if not data_sample:
            print(f"[DataSpinner] not enough samples for '{seed_type}'")
            return 0

        # unpack
        sample_instructions, _ = zip(*[
            (ex["instruction"], ex["json"]) for ex in data_sample["examples"]
        ])

        # 2) generate new instructions
        new_instructions, _ = self.instruction_gen.generate_instructions(
            list(sample_instructions), seed_type, self.prompt_instr, batch_size
        )

        # 3) convert to JSON
        instr_json_pairs, _ = self.json_gen.generate_json(
            new_instructions, data_sample, seed_type, self.prompt_json
        )

        # 4) logic validation
        logic_ok, _ = self.logic_val.validate_logic(
            instr_json_pairs, seed_type, self.prompt_logic
        )
        # 5) syntax validation
        syntax_ok, _ = self.syntax_val.validate_syntax(
            logic_ok, seed_type, self.prompt_syntax
        )
        # 6) supervision
        final_ok, _ = self.supervisor.supervise(
            syntax_ok, seed_type, self.prompt_sup
        )

        # prepare new entries
        new_entries = [
            {"instruction": instr, "json": js if isinstance(js, dict) else js.dict()}
            for instr, js in final_ok
        ]
        if new_entries:
            self.picker.append_new_pairs(seed_type, new_entries)
            print(f"[DataSpinner] appended {len(new_entries)} new pairs to seed_{seed_type}.json")
        else:
            print("[DataSpinner] no valid pairs to append")

        return len(new_entries)

    def run_loop(
        self,
        seed_type: str,
        iterations: int,
        batch_size: int,
        num_examples: int
    ):
        """
        Run multiple iterations of the pipeline with given parameters.
        """
        total = 0
        for i in range(iterations):
            print(f"--- Iteration {i+1}/{iterations} ---")
            added = self.run_once(seed_type, batch_size, num_examples)
            total += added
        print(f"[DataSpinner] done: {total} total new pairs added.")


# 1. Instantiate once (point it at your seed folder)
spinner = DataSpinner(seed_folder="growing_seed")

# 2. Run the pipeline in a loop:
#    • seed_type:    which difficulty to pull from (“easy”, “medium”, “hard”)
#    • iterations:   how many times to run the full pipeline
#    • batch_size:   how many new instructions to generate each iteration
#    • num_examples: how many seed examples to sample each iteration
spinner.run_loop(
    seed_type="easy",
    iterations=5,
    batch_size=20,
    num_examples=3
)

spinner.run_loop(
    seed_type="medium",
    iterations=5,
    batch_size=20,
    num_examples=3
)

spinner.run_loop(
    seed_type="hard",
    iterations=5,
    batch_size=20,
    num_examples=3
)


/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "InstructionJSONPair" shadows an attribute in parent "BaseModel"
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "ValidatedSolution" shadows an attribute in parent "BaseModel"
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "ValidatedSyntax" shadows an attribute in parent "BaseModel"
  warnings.warn(


Newly generated instructions:
- Proceed to the orchard and collect a dozen crates of harvested tomatoes. Transport these crates to the packaging facility and wait until all the crates are unloaded. After completion, travel to the nearby shade area and rest for 10 minutes before heading back to the orchard.
- Navigate to the chemical storage unit to pick up five sealed bottles of fertilizer. Carefully transport them to the greenhouse section and ensure they are unloaded with caution. Then, make your way to the adjacent control station and remain there for 20 minutes for monitoring purposes before returning to the chemical storage.
- Drive to the fuel depot to retrieve a small canister of diesel. Deliver it to the vehicle maintenance bay, ensuring it is stored properly. Once done, proceed to the lunch area and wait there for 15 minutes, then head back to the maintenance bay for further instructions.
- Make your way to the barn to collect a stack of hay bales. Transport these bales to the

/usr/local/lib/python3.10/dist-packages/pydantic/json_schema.py:2279: PydanticJsonSchemaWarning: Default value <function BaseModel.json at 0x7f427da61240> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


Newly generated JSONs:
- ('Proceed to the orchard and collect a dozen crates of harvested tomatoes. Transport these crates to the packaging facility and wait until all the crates are unloaded. After completion, travel to the nearby shade area and rest for 10 minutes before heading back to the orchard.', {'instruction': [{'action': 'go_to', 'location': 'orchard', 'object': None, 'duration': None}, {'action': 'wait_for_payload_ready', 'location': 'orchard', 'object': [{'description': 'crates of harvested tomatoes', 'num': '12'}], 'duration': None}, {'action': 'go_to', 'location': 'packaging facility', 'object': None, 'duration': None}, {'action': 'wait_for_payload_removal', 'location': 'packaging facility', 'object': [{'description': 'crates of harvested tomatoes', 'num': '12'}], 'duration': None}, {'action': 'go_to', 'location': 'shade area', 'object': None, 'duration': None}, {'action': 'wait_for_seconds', 'location': 'shade area', 'object': None, 'duration': 600}, {'action': 'go_to', 